# Logistic Regression

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
assets = ['BTC', 'ETH', 'XRP', 'SOL']

feature_cols = [
    'open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'number_of_trades',
    'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h',
    'ret_6h', 'log_ret_6h', 'ret_12h', 'log_ret_12h',
    'range_pct', 'body_pct', 'upper_shadow_pct', 'lower_shadow_pct', 'close_loc',
    'close_to_sma_12', 'close_to_sma_24',
    'close_to_ema_12', 'close_to_ema_24',
    'vol_24h', 'atr_14_pct', 'rsi_14',
    'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct',
    'log_volume', 'log_quote_volume', 'log_trades',
    'vol_z_24', 'quote_vol_z_24', 'trades_z_24',
    'taker_buy_base_share', 'taker_buy_quote_share', 'order_flow_imbalance',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]

target_col = 'label_up'

train_data, val_data, test_data = {}, {}, {}

for asset in assets:
    train_data[asset] = pd.read_csv(f"data/split_data/{asset}_train.csv")
    val_data[asset]   = pd.read_csv(f"data/split_data/{asset}_val.csv")
    test_data[asset]  = pd.read_csv(f"data/split_data/{asset}_test.csv")


## Logistic Regression

In [3]:
def run_logistic_regression(train_df, val_df, asset_name, feature_cols, target_col):
    usable_features = [col for col in feature_cols if col in train_df.columns and col in val_df.columns]

    train_df = train_df.dropna(subset=usable_features + [target_col]).copy()
    val_df = val_df.dropna(subset=usable_features + [target_col]).copy()

    X_train = train_df[usable_features]
    y_train = train_df[target_col]
    X_val = val_df[usable_features]
    y_val = val_df[target_col]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)
    y_prob = model.predict_proba(X_val_scaled)[:, 1]

    acc = accuracy_score(y_val, y_pred)

    print(f"\n--- Logistic Regression: {asset_name} ---")
    print(f"Validation Accuracy: {acc:.4f}")
    print(confusion_matrix(y_val, y_pred))
    print(classification_report(y_val, y_pred))

    return {
        'model': model,
        'scaler': scaler,
        'features_used': usable_features,
        'accuracy': acc,
        'y_val': y_val,
        'y_pred': y_pred,
        'y_prob': y_prob
    }

In [4]:
results = {}
for asset in assets:
    results[asset] = run_logistic_regression(
        train_data[asset],
        val_data[asset],
        asset,
        feature_cols,
        target_col
    )


--- Logistic Regression: BTC ---
Validation Accuracy: 0.5442
[[3189 2169]
 [2815 2761]]
              precision    recall  f1-score   support

           0       0.53      0.60      0.56      5358
           1       0.56      0.50      0.53      5576

    accuracy                           0.54     10934
   macro avg       0.55      0.55      0.54     10934
weighted avg       0.55      0.54      0.54     10934


--- Logistic Regression: ETH ---
Validation Accuracy: 0.5436
[[2266 3128]
 [1862 3678]]
              precision    recall  f1-score   support

           0       0.55      0.42      0.48      5394
           1       0.54      0.66      0.60      5540

    accuracy                           0.54     10934
   macro avg       0.54      0.54      0.54     10934
weighted avg       0.54      0.54      0.54     10934


--- Logistic Regression: XRP ---
Validation Accuracy: 0.5399
[[3118 2338]
 [2693 2785]]
              precision    recall  f1-score   support

           0       0.54 

## Tune Probability Threshold Using Sharpe Ratio

In [5]:
import numpy as np

def tune_threshold_sharpe(y_prob, val_df, thresholds, cost=0.00035):

    best_sharpe = -np.inf
    best_threshold = None

    sharpe_list = []

    for t in thresholds:
        signal = (y_prob >= t).astype(int)

        # strategy returns
        ret = signal * val_df['fwd_ret_open_1h']

        # transaction cost
        turnover = np.abs(np.diff(signal, prepend=0))
        ret = ret - turnover * cost

        # avoid division by zero
        if ret.std() == 0:
            sharpe = 0
        else:
            sharpe = ret.mean() / ret.std() * np.sqrt(8760)

        sharpe_list.append((t, sharpe))

        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_threshold = t

    return best_threshold, best_sharpe, sharpe_list

In [6]:
threshold_grid = np.arange(0.45, 0.65, 0.01)

threshold_results = {}

for asset in assets:
    y_prob = results[asset]['y_prob']
    val_df = val_data[asset].loc[results[asset]['y_val'].index]

    best_t, best_sharpe, sharpe_curve = tune_threshold_sharpe(
        y_prob, val_df, threshold_grid
    )

    print(f"\n{asset}")
    print(f"Best threshold: {best_t}")
    print(f"Best Sharpe: {best_sharpe:.4f}")

    threshold_results[asset] = {
        'best_threshold': best_t,
        'best_sharpe': best_sharpe,
        'curve': sharpe_curve
    }


BTC
Best threshold: 0.6300000000000001
Best Sharpe: 0.8406

ETH
Best threshold: 0.6400000000000001
Best Sharpe: 1.3782

XRP
Best threshold: 0.5800000000000001
Best Sharpe: 0.9547

SOL
Best threshold: 0.54
Best Sharpe: 2.3459


In [7]:
for asset in assets:
    print(asset, threshold_results[asset]['best_threshold'])

BTC 0.6300000000000001
ETH 0.6400000000000001
XRP 0.5800000000000001
SOL 0.54


The decision threshold is calibrated on the validation set by maximising the annualised Sharpe ratio under realistic transaction costs. This ensures that model outputs are translated into economically meaningful trading signals rather than purely statistical classifications.

## Tune Regularisation Strength

In [8]:
def tune_C_and_threshold(train_df, val_df, asset, feature_cols, target_col):

    C_grid = [0.01, 0.1, 1, 10]
    threshold_grid = np.arange(0.45, 0.65, 0.01)

    best_result = {
        'C': None,
        'threshold': None,
        'sharpe': -np.inf
    }

    usable_features = [col for col in feature_cols if col in train_df.columns and col in val_df.columns]

    train_df = train_df.dropna(subset=usable_features + [target_col]).copy()
    val_df = val_df.dropna(subset=usable_features + [target_col]).copy()

    X_train = train_df[usable_features]
    y_train = train_df[target_col]
    X_val = val_df[usable_features]
    y_val = val_df[target_col]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    for C in C_grid:
        model = LogisticRegression(max_iter=2000, random_state=42, C=C)
        model.fit(X_train_scaled, y_train)

        y_prob = model.predict_proba(X_val_scaled)[:, 1]

        # align index
        val_df_aligned = val_df.loc[y_val.index]

        best_t, best_sharpe, _ = tune_threshold_sharpe(
            y_prob, val_df_aligned, threshold_grid
        )

        if best_sharpe > best_result['sharpe']:
            best_result = {
                'C': C,
                'threshold': best_t,
                'sharpe': best_sharpe,
                'model': model,
                'scaler': scaler,
                'features': usable_features
            }

    print(f"\n{asset} BEST:")
    print(best_result)

    return best_result

In [9]:
tuned_results = {}

for asset in assets:
    tuned_results[asset] = tune_C_and_threshold(
        train_data[asset],
        val_data[asset],
        asset,
        feature_cols,
        target_col
    )


BTC BEST:
{'C': 10, 'threshold': 0.6300000000000001, 'sharpe': 0.8417412217145916, 'model': LogisticRegression(C=10, max_iter=2000, random_state=42), 'scaler': StandardScaler(), 'features': ['open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'number_of_trades', 'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h', 'ret_6h', 'log_ret_6h', 'ret_12h', 'log_ret_12h', 'range_pct', 'body_pct', 'upper_shadow_pct', 'lower_shadow_pct', 'close_loc', 'close_to_sma_12', 'close_to_sma_24', 'close_to_ema_12', 'close_to_ema_24', 'vol_24h', 'atr_14_pct', 'rsi_14', 'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct', 'log_volume', 'log_quote_volume', 'log_trades', 'vol_z_24', 'quote_vol_z_24', 'trades_z_24', 'taker_buy_base_share', 'taker_buy_quote_share', 'order_flow_imbalance', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']}

ETH BEST:
{'C': 10, 'threshold': 0.6400000000000001, 'sharpe': 1.3869461891744035, 'model': LogisticRegression(C=10, max_iter=2000, random_state=42), 'scaler'

In [10]:
for asset in assets:
    print(asset, tuned_results[asset]['C'], tuned_results[asset]['threshold'], tuned_results[asset]['sharpe'])

BTC 10 0.6300000000000001 0.8417412217145916
ETH 10 0.6400000000000001 1.3869461891744035
XRP 10 0.6000000000000001 1.1646047229057093
SOL 10 0.52 2.399454122906774


While regularization strength was tuned, the resulting improvements in Sharpe ratio were modest across most assets. This suggests that model performance is primarily driven by the decision threshold used to convert predicted probabilities into trading signals, rather than the specific regularization parameter of the logistic regression model.

## Backtesting

In [11]:
import numpy as np
import pandas as pd
from performance_utils import summarize_performance, compute_directional_metrics

def backtest_logistic_regression(
    test_df: pd.DataFrame,
    model,
    scaler,
    features: list,
    threshold: float,
    return_col: str = "fwd_ret_open_1h",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035
) -> pd.DataFrame:
    """
    Backtest a logistic regression long-flat strategy using precomputed forward open-to-open returns.

    Strategy logic:
    - Predict probability of upward movement using fitted logistic regression
    - Go long (position = 1) if probability >= threshold
    - Otherwise stay out (position = 0)
    - Use precomputed forward return column for realized next-period return
    - Apply transaction cost whenever position changes

    Parameters
    ----------
    test_df : pd.DataFrame
        Test-period data.
    model : fitted sklearn LogisticRegression
        Trained logistic regression model.
    scaler : fitted sklearn StandardScaler
        Scaler fitted on training data.
    features : list
        Feature columns used by the model.
    threshold : float
        Fixed probability threshold chosen from validation.
    return_col : str
        Column containing forward open-to-open return.
    timestamp_col : str, optional
        Optional timestamp column for readability.
    transaction_cost : float
        Proportional cost per trade (e.g. 0.00035 = 0.035%).

    Returns
    -------
    pd.DataFrame
        Backtest dataframe with probabilities, positions, costs, net returns, and equity curve.
    """
    df = test_df.copy().reset_index(drop=True)

    # Keep only rows with complete data needed for prediction and return realization
    required_cols = features + [return_col]
    if timestamp_col is not None and timestamp_col in df.columns:
        required_cols = [timestamp_col] + required_cols

    df = df.dropna(subset=required_cols).copy().reset_index(drop=True)

    # Generate predicted probabilities
    X_test = df[features]
    X_test_scaled = scaler.transform(X_test)
    df["y_prob"] = model.predict_proba(X_test_scaled)[:, 1]

    # Realized next-period forward return
    df["raw_return"] = df[return_col]

    # Position: long if probability exceeds threshold, else flat
    df["position"] = (df["y_prob"] >= threshold).astype(int)

    # Position changes determine transaction costs
    df["prev_position"] = df["position"].shift(1).fillna(0)
    df["position_change"] = (df["position"] - df["prev_position"]).abs()

    # Transaction cost whenever position changes
    df["transaction_cost"] = transaction_cost * df["position_change"]

    # Gross and net strategy returns
    df["gross_return"] = df["position"] * df["raw_return"]
    df["net_return"] = df["gross_return"] - df["transaction_cost"]

    # Equity curves
    df["gross_equity_curve"] = (1 + df["gross_return"]).cumprod()
    df["net_equity_curve"] = (1 + df["net_return"]).cumprod()

    return df

In [12]:
def run_logistic_regression_backtest(
    test_df: pd.DataFrame,
    model,
    scaler,
    features: list,
    threshold: float,
    return_col: str = "fwd_ret_open_1h",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035,
    periods_per_year: int = 8760,
    risk_free_rate: float = 0.0,
    include_directional_metrics: bool = True
):
    """
    Full pipeline:
    1. Run logistic regression backtest
    2. Compute performance summary
    3. Optionally compute directional metrics
    """
    backtest_df = backtest_logistic_regression(
        test_df=test_df,
        model=model,
        scaler=scaler,
        features=features,
        threshold=threshold,
        return_col=return_col,
        timestamp_col=timestamp_col,
        transaction_cost=transaction_cost
    )

    performance_summary = summarize_performance(
        backtest_df=backtest_df,
        return_col="net_return",
        position_col="position",
        periods_per_year=periods_per_year,
        risk_free_rate=risk_free_rate
    )

    if include_directional_metrics:
        directional_summary = compute_directional_metrics(backtest_df)
        full_summary = pd.concat([performance_summary, directional_summary])
    else:
        full_summary = performance_summary

    return backtest_df, full_summary

### Results

In [13]:
btc_test = pd.read_csv("data/split_data/BTC_test.csv")
eth_test = pd.read_csv("data/split_data/ETH_test.csv")
xrp_test = pd.read_csv("data/split_data/XRP_test.csv")
sol_test = pd.read_csv("data/split_data/SOL_test.csv")

test_datasets = {
    "BTC": btc_test,
    "ETH": eth_test,
    "XRP": xrp_test,
    "SOL": sol_test
}

all_lr_backtest_results = {}
all_lr_summaries = []

for asset_name, test_df in test_datasets.items():
    model = tuned_results[asset_name]["model"]
    scaler = tuned_results[asset_name]["scaler"]
    features = tuned_results[asset_name]["features"]
    threshold = tuned_results[asset_name]["threshold"]

    backtest_results, summary = run_logistic_regression_backtest(
        test_df=test_df,
        model=model,
        scaler=scaler,
        features=features,
        threshold=threshold,
        return_col="fwd_ret_open_1h",
        timestamp_col="timestamp",   # change to "datetime" if needed
        transaction_cost=0.00035,
        periods_per_year=8760,
        risk_free_rate=0.0,
        include_directional_metrics=True
    )

    all_lr_backtest_results[asset_name] = backtest_results

    summary_df = summary.to_frame().T
    summary_df.insert(0, "Asset", asset_name)
    summary_df.insert(1, "Model", "Logistic Regression")
    summary_df.insert(2, "C", tuned_results[asset_name]["C"])
    summary_df.insert(3, "Threshold", tuned_results[asset_name]["threshold"])
    all_lr_summaries.append(summary_df)

final_lr_summary_table = pd.concat(all_lr_summaries, ignore_index=True)

print("===== LOGISTIC REGRESSION SUMMARY FOR ALL ASSETS =====")
print(final_lr_summary_table)

===== LOGISTIC REGRESSION SUMMARY FOR ALL ASSETS =====
  Asset                Model   C  Threshold  Cumulative Return  \
0   BTC  Logistic Regression  10       0.63           0.001382   
1   ETH  Logistic Regression  10       0.64           0.120662   
2   XRP  Logistic Regression  10       0.60           0.924020   
3   SOL  Logistic Regression  10       0.52           0.013171   

   Annualized Return  Annualized Volatility  Annualized Sharpe  \
0           0.001107               0.060536           0.048434   
1           0.095563               0.107514           0.902797   
2           0.689275               0.263144           2.123156   
3           0.010538               0.581339           0.308507   

   Maximum Drawdown  Turnover  Number of Trades  Average Return per Trade  \
0         -0.062790     132.0             132.0                  0.000404   
1         -0.061007     520.0             520.0                  0.000815   
2         -0.170835     342.0             342.0     

In [15]:
# save results
for asset_name, df in all_lr_backtest_results.items():
    df.to_csv(f"result/logistic_regression/{asset_name.lower()}_backtest.csv", index=False)

final_lr_summary_table.to_csv("result/logistic_regression/summary_all_assets.csv", index=False)

The logistic regression strategy demonstrates heterogeneous performance across assets. While it fails to outperform buy-and-hold for BTC, it significantly improves risk-adjusted performance for ETH and XRP, with Sharpe ratios increasing from 0.56 to 0.90 and 1.41 to 2.12 respectively.

Notably, for XRP, the model achieves both strong returns and superior Sharpe ratio, indicating the presence of exploitable short-term patterns.

In contrast, for SOL, the model reduces losses relative to buy-and-hold but suffers from high turnover and transaction costs, limiting overall performance.

These results highlight that simple linear models can extract economically meaningful signals in certain markets, but their effectiveness is asset-dependent.
